# SWE015 Homework: Cargo Support Ticket Router (Use Case 2)
**Student:** Eren Acar (`2309111059`)  


In [ ]:
!pip install -q transformers gensim scikit-learn gradio torch

In [ ]:
# Write preprocessing.py, dataset.py and train_models.py (same content as the repository files)
with open('preprocessing.py', 'w') as f:
    f.write('import re\n\n# Domain word list for spelling correction\nCARGO_WORDS = [\n    "parcel", "package", "shipment", "delivery", "courier", "driver",\n    "tracking", "delayed", "stuck", "sorting", "warehouse", "branch",\n    "damaged", "broken", "crushed", "missing", "lost", "address",\n    "return", "refund", "cancel", "customs", "insurance", "receipt"\n]\n\n# ELIZA-style rule responses for small talk and greetings\nSMALL_TALK_RULES = [\n    (r"^(hi|hello|hey|good morning|good afternoon)\\b", "Hello! Welcome to Cargo Support. Please share your tracking code or issue."),\n    (r"how are you", "I am doing well, thank you! How can I help you with your shipment today?"),\n    (r"(who are you|are you a bot)", "I am the automated Cargo Support Ticket Router built for SWE015."),\n    (r"(thank you|thanks)", "You are welcome! Let me know if you need more help with your parcel.")\n]\n\n\ndef edit_distance(word1, word2):\n    """Standard dynamic programming Levenshtein minimum edit distance."""\n    m, n = len(word1), len(word2)\n    dp = [[0] * (n + 1) for _ in range(m + 1)]\n\n    for i in range(m + 1):\n        dp[i][0] = i\n    for j in range(n + 1):\n        dp[0][j] = j\n\n    for i in range(1, m + 1):\n        for j in range(1, n + 1):\n            cost = 0 if word1[i - 1] == word2[j - 1] else 1\n            dp[i][j] = min(\n                dp[i - 1][j] + 1,      # deletion\n                dp[i][j - 1] + 1,      # insertion\n                dp[i - 1][j - 1] + cost  # substitution\n            )\n    return dp[m][n]\n\n\ndef correct_spelling(word):\n    """Finds the closest cargo domain word if edit distance is 1 or 2."""\n    if len(word) < 4 or not word.isalpha() or word in CARGO_WORDS:\n        return word, False\n\n    best_word = word\n    min_dist = 3\n\n    for target in CARGO_WORDS:\n        # Only compare words that start with the same letter and have similar length\n        if word[0] != target[0] or abs(len(word) - len(target)) > 2:\n            continue\n        dist = edit_distance(word, target)\n        allowed = 1 if len(target) <= 6 else 2\n        if dist <= allowed and dist < min_dist:\n            min_dist = dist\n            best_word = target\n\n    return best_word, (best_word != word)\n\n\nCODE_PATTERN = r"\\b(?:TRK|ORD|SHP|BR)-?\\d{3,8}\\b"\n\n\ndef check_small_talk(text):\n    """\n    Checks if the message is ONLY a greeting / small talk (ELIZA rule).\n    A short message that also contains a tracking code or a cargo keyword\n    (e.g. "hi, my parcel is lost") is a real ticket and must NOT be swallowed.\n    """\n    clean = text.strip().lower()\n    if len(clean.split()) > 6:\n        return None\n    if re.search(CODE_PATTERN, text, flags=re.IGNORECASE):\n        return None\n    if any(re.search(rf"\\b{w}", clean) for w in CARGO_WORDS):\n        return None\n    for pattern, reply in SMALL_TALK_RULES:\n        if re.search(pattern, clean):\n            return reply\n    return None\n\n\ndef preprocess_message(raw_text):\n    """\n    Runs Area B preprocessing:\n    1. Checks ELIZA small talk rules\n    2. Extracts tracking / order codes with regex (e.g. TRK-12345, ORD-9876)\n    3. Normalizes text and fixes typos using minimum edit distance\n    """\n    small_talk_reply = check_small_talk(raw_text)\n\n    # Extract tracking and order codes like TRK-84920 or ORD-1029\n    codes = re.findall(CODE_PATTERN, raw_text.upper())\n    text_no_codes = re.sub(CODE_PATTERN, " ", raw_text, flags=re.IGNORECASE)\n\n    # Lowercase and remove special characters\n    normalized = re.sub(r"(.)\\1{2,}", r"\\1", text_no_codes.lower())\n    normalized = re.sub(r"[^a-z0-9\\s]", " ", normalized)\n    words = normalized.split()\n\n    corrected_words = []\n    corrections = []\n    for w in words:\n        fixed, changed = correct_spelling(w)\n        corrected_words.append(fixed)\n        if changed:\n            corrections.append(f"{w} -> {fixed}")\n\n    clean_text = " ".join(corrected_words)\n\n    return {\n        "raw_text": raw_text,\n        "clean_text": clean_text,\n        "tokens": corrected_words,\n        "extracted_codes": codes,\n        "corrections": corrections,\n        "small_talk_reply": small_talk_reply\n    }\n')
with open('dataset.py', 'w') as f:
    f.write('"""\nSynthetic dataset for the Cargo Support Ticket Router (self-generated, template based).\n\nThe held-out TEST set is built ONLY from templates that never appear in train/validation,\nso test messages have sentence structures the models have not seen during training.\n"""\n\nimport random\n\nINTENT_NAMES = [\n    "delayed_delivery",\n    "damaged_or_missing",\n    "courier_complaint",\n    "address_or_branch",\n    "return_and_refund",\n    "out_of_scope"\n]\n\nTEAM_MAP = {\n    "delayed_delivery": "Hub Operations & Delayed Delivery Team",\n    "damaged_or_missing": "Damaged & Lost Parcel Claims Team",\n    "courier_complaint": "Courier Field Audit & Quality Team",\n    "address_or_branch": "Address Update & Branch Rerouting Team",\n    "return_and_refund": "Returns & Refund Logistics Team",\n    "out_of_scope": "Human Agent (Out-of-Scope / Escalation)"\n}\n\n# Last N_TEST_TEMPLATES templates of every intent are used ONLY for the test set.\nN_TEST_TEMPLATES = 2\n\nTEMPLATES = {\n    "delayed_delivery": [\n        "my parcel {code} has been stuck at the sorting center for {days} days",\n        "shipment {code} is delayed and passed the expected delivery date",\n        "why is my package {code} still waiting at the warehouse for {days} days",\n        "my express delivery {code} has not moved since last week",\n        "parcel {code} is still in transit after {days} days with no update",\n        "tracking shows my shipment {code} is delayed at the main hub",\n        "my order {code} should have arrived {days} days ago but it is still on the way",\n        "there has been no tracking movement on package {code} for {days} days",\n        # --- held-out (test only) ---\n        "i am still waiting for parcel {code} and it is already {days} days late",\n        "the estimated arrival date for shipment {code} keeps getting pushed back",\n    ],\n    "damaged_or_missing": [\n        "my package {code} arrived completely crushed and the item inside is broken",\n        "the box for parcel {code} was torn open and one item is missing",\n        "shipment {code} says delivered but nothing was left at my door it is lost",\n        "my delivery {code} arrived wet and damaged i want to file a claim",\n        "the glass inside my parcel {code} was shattered during shipping",\n        "package {code} is missing from my porch even though status says delivered",\n        "the contents of parcel {code} were smashed when i opened the box",\n        "i never received shipment {code} and i think it was lost on the way",\n        # --- held-out (test only) ---\n        "my parcel {code} showed up dented and the product is ruined",\n        "half of the things in package {code} are gone and the box was open",\n    ],\n    "courier_complaint": [\n        "the courier never rang the doorbell and left a missed delivery note for {code}",\n        "delivery driver for {code} was very rude on the phone and threw my parcel",\n        "the driver refused to bring package {code} up to my apartment door",\n        "courier marked my shipment {code} as customer not home while i was waiting inside",\n        "the delivery person left parcel {code} in the rain without knocking",\n        "i want to report the courier who handled my shipment {code} today",\n        "the driver shouted at me when i asked about parcel {code}",\n        "courier did not even try to call me before returning package {code}",\n        # --- held-out (test only) ---\n        "the man who brought shipment {code} was impolite and careless with my box",\n        "i am unhappy with how the delivery guy treated me for order {code}",\n    ],\n    "address_or_branch": [\n        "please change the delivery address for parcel {code} to my office street",\n        "i want to pick up shipment {code} at the local branch instead of home delivery",\n        "my package {code} was sent to the wrong branch please reroute it",\n        "can i update the apartment number and zip code for delivery {code}",\n        "hold my parcel {code} at the nearest branch office for self pickup",\n        "i entered the wrong street address for order {code} please fix it",\n        "i moved to a new house so please redirect shipment {code} there",\n        "can you deliver package {code} to my neighbour address instead",\n        # --- held-out (test only) ---\n        "send parcel {code} to a different city because i will be travelling",\n        "i would rather collect order {code} from a pickup point near my work",\n    ],\n    "return_and_refund": [\n        "i want to return my order {code} and get a return shipping label",\n        "please cancel shipment {code} and send the package back to the seller",\n        "when will the courier pick up my return parcel for order {code}",\n        "i returned package {code} last week when will i get my shipping refund",\n        "how do i schedule a home pickup to return damaged order {code}",\n        "need a free return code and refund for my shipment {code}",\n        "i changed my mind about order {code} and want my money back",\n        "where do i drop off the parcel {code} i am sending back",\n        # --- held-out (test only) ---\n        "the refund for my returned shipment {code} has not reached my account yet",\n        "i do not want package {code} anymore so how can i give it back",\n    ],\n    "out_of_scope": [\n        "what is the weather forecast for london this weekend",\n        "my washing machine is making a loud noise during the spin cycle",\n        "can you recommend a good laptop for university programming classes",\n        "i want to book a hotel room and flight ticket for next month",\n        "how do i reset my wifi router password at home",\n        "what time does the football match start on television tonight",\n        "give me a simple recipe for chicken soup",\n        "who won the basketball game last night",\n        # --- held-out (test only) ---\n        "how can i learn to play the guitar quickly",\n        "what is the best movie to watch on a rainy day",\n    ],\n}\n\nTYPO_SAMPLES = {\n    "parcel": ["parcel", "parcl", "parsel"],\n    "delivery": ["delivery", "dlivery", "delivry"],\n    "courier": ["courier", "coureir", "courer"],\n    "damaged": ["damaged", "damagd", "damged"],\n    "tracking": ["tracking", "traking", "trackng"]\n}\n\nN_PER_INTENT = 150\nN_TEST_PER_INTENT = 30\n\n\ndef _make_example(rng, tpl, i):\n    code = f"TRK-{rng.randint(1000, 9999)}"\n    days = rng.randint(3, 10)\n    text = tpl.format(code=code, days=days)\n\n    # Occasional typos (25% chance) to exercise the spelling corrector\n    if rng.random() < 0.25:\n        for clean_w, typos in TYPO_SAMPLES.items():\n            if clean_w in text:\n                text = text.replace(clean_w, rng.choice(typos), 1)\n\n    # Mild conversational noise\n    if i % 5 == 0:\n        text += rng.choice([\n            " please help urgently",\n            " i tried calling support earlier",\n            " this is my second message today",\n            " thank you for checking"\n        ])\n    return text\n\n\ndef load_dataset(seed=42):\n    """\n    Returns train / validation / held-out test splits.\n    train+val  : first (len(templates) - N_TEST_TEMPLATES) templates   (120 per intent -> 100 / 20)\n    test       : last  N_TEST_TEMPLATES templates, never seen in training (30 per intent)\n    """\n    rng = random.Random(seed)\n    train_val, test_data = [], []\n\n    for label_id, intent in enumerate(INTENT_NAMES):\n        templates = TEMPLATES[intent]\n        seen, unseen = templates[:-N_TEST_TEMPLATES], templates[-N_TEST_TEMPLATES:]\n\n        for i in range(N_PER_INTENT - N_TEST_PER_INTENT):\n            text = _make_example(rng, seen[i % len(seen)], i)\n            train_val.append({"text": text, "intent": intent, "label": label_id})\n        for i in range(N_TEST_PER_INTENT):\n            text = _make_example(rng, unseen[i % len(unseen)], i)\n            test_data.append({"text": text, "intent": intent, "label": label_id})\n\n    rng.shuffle(train_val)\n    rng.shuffle(test_data)\n    n_val = int(0.15 * len(train_val))\n    val_data = train_val[:n_val]\n    train_data = train_val[n_val:]\n    return train_data, val_data, test_data\n')
with open('train_models.py', 'w') as f:
    f.write('"""\nSWE015 Homework - Training & Evaluation Script for the 4-Model Ladder:\n1. Model 1 (Classical Baseline - Area B): TF-IDF + Logistic Regression\n2. Model 2 (Neural Network - Area A): Word2Vec + Feedforward NN trained with SGD & Backprop\n3. Model 3 (Transformer - Area C): Fine-Tuned BERT (bert-base-uncased)\n4. Model 4 (Quantized Transformer - Area D): Distilled Student (distilbert-base-uncased) + Dynamic INT8 Quantization\n"""\n\nimport json\nimport os\nimport pickle\nimport platform\nimport time\nfrom pathlib import Path\n\nimport numpy as np\nimport torch\nimport torch.nn as nn\nimport torch.nn.functional as F\nfrom torch.utils.data import DataLoader, Dataset\nfrom sklearn.feature_extraction.text import TfidfVectorizer\nfrom sklearn.linear_model import LogisticRegression\nfrom sklearn.metrics import accuracy_score, f1_score, recall_score\nfrom gensim.models import Word2Vec\nfrom transformers import AutoModelForSequenceClassification, AutoTokenizer\n\nfrom dataset import INTENT_NAMES, load_dataset\nfrom preprocessing import preprocess_message\n\nMODELS_DIR = Path("models")\nMODELS_DIR.mkdir(exist_ok=True)\nDEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")\n\n\n# --- Area A: Simple Feedforward Neural Network ---\nclass SimpleFFNN(nn.Module):\n    def __init__(self, input_dim=100, hidden_dim=64, num_classes=6):\n        super().__init__()\n        self.fc1 = nn.Linear(input_dim, hidden_dim)\n        self.relu = nn.ReLU()\n        self.dropout = nn.Dropout(0.2)\n        self.fc2 = nn.Linear(hidden_dim, num_classes)\n\n    def forward(self, x):\n        x = self.relu(self.fc1(x))\n        x = self.dropout(x)\n        return self.fc2(x)\n\n\ndef sentence_vector(tokens, w2v_model, dim=100):\n    vecs = [w2v_model.wv[w] for w in tokens if w in w2v_model.wv]\n    if len(vecs) == 0:\n        return np.zeros(dim, dtype=np.float32)\n    return np.mean(vecs, axis=0).astype(np.float32)\n\n\n# --- Simple PyTorch Dataset for BERT / DistilBERT ---\nclass TicketDataset(Dataset):\n    def __init__(self, texts, labels, tokenizer, max_len=48):\n        self.encodings = tokenizer(texts, truncation=True, padding="max_length", max_length=max_len, return_tensors="pt")\n        self.labels = torch.tensor(labels, dtype=torch.long)\n\n    def __len__(self):\n        return len(self.labels)\n\n    def __getitem__(self, idx):\n        return {\n            "input_ids": self.encodings["input_ids"][idx],\n            "attention_mask": self.encodings["attention_mask"][idx],\n            "labels": self.labels[idx]\n        }\n\n\ndef measure_metrics_and_latency(predict_fn, test_texts, y_true, model_file_path, n_runs=100):\n    preds = [predict_fn(t) for t in test_texts]\n    acc = accuracy_score(y_true, preds) * 100\n    f1 = f1_score(y_true, preds, average="macro") * 100\n\n    # Out-of-scope class is index 5\n    oos_id = INTENT_NAMES.index("out_of_scope")\n    oos_true = [1 if y == oos_id else 0 for y in y_true]\n    oos_pred = [1 if p == oos_id else 0 for p in preds]\n    oos_recall = recall_score(oos_true, oos_pred, zero_division=0) * 100\n\n    # Measure average latency over 100 runs on a sample ticket\n    sample_text = test_texts[0]\n    for _ in range(5):\n        predict_fn(sample_text)\n    start = time.perf_counter()\n    for _ in range(n_runs):\n        predict_fn(sample_text)\n    avg_latency_ms = ((time.perf_counter() - start) / n_runs) * 1000\n\n    # Measure size on disk in MB\n    if os.path.isdir(model_file_path):\n        total_bytes = sum(f.stat().st_size for f in Path(model_file_path).glob("**/*") if f.is_file())\n    else:\n        total_bytes = os.path.getsize(model_file_path)\n    size_mb = total_bytes / (1024 * 1024)\n\n    return round(acc, 2), round(oos_recall, 2), round(f1, 2), round(size_mb, 2), round(avg_latency_ms, 2)\n\n\ndef hardware_description():\n    """CPU model + threads + library versions (R4 asks to state the hardware)."""\n    cpu = platform.processor()\n    try:\n        with open("/proc/cpuinfo") as f:\n            for line in f:\n                if line.startswith("model name"):\n                    cpu = line.split(":", 1)[1].strip()\n                    break\n    except OSError:\n        pass\n    return (f"{platform.system()} | CPU: {cpu or \'unknown\'} | torch threads: {torch.get_num_threads()} | "\n            f"torch {torch.__version__} | inference on CPU")\n\n\ndef main():\n    print(f"Using device: {DEVICE}")\n    train_raw, val_raw, test_raw = load_dataset(seed=42)\n\n    train_prep = [preprocess_message(x["text"]) for x in train_raw]\n    test_prep = [preprocess_message(x["text"]) for x in test_raw]\n\n    X_train_text = [x["clean_text"] for x in train_prep]\n    X_train_tokens = [x["tokens"] for x in train_prep]\n    y_train = [x["label"] for x in train_raw]\n\n    X_test_text = [x["clean_text"] for x in test_prep]\n    X_test_tokens = [x["tokens"] for x in test_prep]\n    y_test = [x["label"] for x in test_raw]\n\n    # =========================================================================\n    # 1. Model 1: Classical Baseline (TF-IDF + Logistic Regression)\n    # =========================================================================\n    print("Training Model 1: TF-IDF + Logistic Regression...")\n    vectorizer = TfidfVectorizer(max_features=1000)\n    X_tr_vec = vectorizer.fit_transform(X_train_text)\n    clf = LogisticRegression(max_iter=300)\n    clf.fit(X_tr_vec, y_train)\n\n    tfidf_path = MODELS_DIR / "tfidf_baseline.pkl"\n    with open(tfidf_path, "wb") as f:\n        pickle.dump({"vectorizer": vectorizer, "clf": clf}, f)\n\n    val_prep = [preprocess_message(x["text"]) for x in val_raw]\n    val_acc = accuracy_score([x["label"] for x in val_raw], clf.predict(vectorizer.transform([x["clean_text"] for x in val_prep]))) * 100\n    print(f"Model 1 validation accuracy: {val_acc:.2f}%")\n\n    # =========================================================================\n    # 2. Model 2: Word2Vec + Feedforward Neural Network (SGD & Backprop)\n    # =========================================================================\n    print("Training Model 2: Word2Vec + SGD Feedforward Neural Network...")\n    w2v = Word2Vec(sentences=X_train_tokens, vector_size=100, window=4, min_count=1, epochs=25, seed=42)\n    w2v_path = MODELS_DIR / "word2vec.model"\n    w2v.save(str(w2v_path))\n\n    X_tr_w2v = torch.tensor(np.array([sentence_vector(t, w2v) for t in X_train_tokens]), dtype=torch.float32)\n    y_tr_tensor = torch.tensor(y_train, dtype=torch.long)\n\n    ffnn = SimpleFFNN(input_dim=100, hidden_dim=64, num_classes=len(INTENT_NAMES))\n    optimizer_sgd = torch.optim.SGD(ffnn.parameters(), lr=0.05, momentum=0.9)\n    criterion = nn.CrossEntropyLoss()\n\n    # Mini-batch SGD (batch size 32) with momentum + backpropagation\n    torch.manual_seed(42)\n    ffnn.train()\n    n_tr = X_tr_w2v.shape[0]\n    for epoch in range(40):\n        perm = torch.randperm(n_tr)\n        for start in range(0, n_tr, 32):\n            idx = perm[start:start + 32]\n            optimizer_sgd.zero_grad()\n            loss = criterion(ffnn(X_tr_w2v[idx]), y_tr_tensor[idx])\n            loss.backward()\n            optimizer_sgd.step()\n\n    ffnn.eval()\n    ffnn_path = MODELS_DIR / "ffnn_sgd.pt"\n    torch.save(ffnn.state_dict(), ffnn_path)\n\n    # =========================================================================\n    # 3. Model 3: Fine-Tuned BERT (Teacher Model)\n    # =========================================================================\n    print("Training Model 3: Fine-Tuning BERT (bert-base-uncased)...")\n    teacher_name = "bert-base-uncased"\n    tokenizer = AutoTokenizer.from_pretrained(teacher_name)\n    teacher_model = AutoModelForSequenceClassification.from_pretrained(teacher_name, num_labels=len(INTENT_NAMES)).to(DEVICE)\n\n    train_ds = TicketDataset(X_train_text, y_train, tokenizer)\n    train_loader = DataLoader(train_ds, batch_size=16, shuffle=True)\n    opt_bert = torch.optim.AdamW(teacher_model.parameters(), lr=3e-5)\n\n    teacher_model.train()\n    for epoch in range(3):\n        for batch in train_loader:\n            opt_bert.zero_grad()\n            inp = batch["input_ids"].to(DEVICE)\n            mask = batch["attention_mask"].to(DEVICE)\n            lbl = batch["labels"].to(DEVICE)\n            outputs = teacher_model(input_ids=inp, attention_mask=mask, labels=lbl)\n            outputs.loss.backward()\n            opt_bert.step()\n\n    teacher_model.eval()\n    teacher_dir = MODELS_DIR / "bert_teacher"\n    teacher_model.save_pretrained(teacher_dir)\n    tokenizer.save_pretrained(teacher_dir)\n\n    # =========================================================================\n    # 4. Model 4: Knowledge Distillation + Dynamic INT8 Quantization\n    # =========================================================================\n    print("Training Model 4: Knowledge Distillation into DistilBERT + INT8 Quantization...")\n    student_name = "distilbert-base-uncased"\n    student_model = AutoModelForSequenceClassification.from_pretrained(student_name, num_labels=len(INTENT_NAMES)).to(DEVICE)\n    opt_student = torch.optim.AdamW(student_model.parameters(), lr=5e-5)\n    kl_loss = nn.KLDivLoss(reduction="batchmean")\n    temp = 2.5\n    alpha = 0.6\n\n    student_model.train()\n    for epoch in range(3):\n        for batch in train_loader:\n            opt_student.zero_grad()\n            inp = batch["input_ids"].to(DEVICE)\n            mask = batch["attention_mask"].to(DEVICE)\n            lbl = batch["labels"].to(DEVICE)\n\n            with torch.no_grad():\n                teacher_logits = teacher_model(input_ids=inp, attention_mask=mask).logits\n\n            student_out = student_model(input_ids=inp, attention_mask=mask, labels=lbl)\n            student_logits = student_out.logits\n\n            # Knowledge distillation loss = soft KL loss + hard Cross-Entropy loss\n            loss_soft = kl_loss(\n                F.log_softmax(student_logits / temp, dim=-1),\n                F.softmax(teacher_logits / temp, dim=-1)\n            ) * (temp ** 2)\n            loss = alpha * loss_soft + (1 - alpha) * student_out.loss\n            loss.backward()\n            opt_student.step()\n\n    # Move models to CPU for fair latency comparison and Dynamic INT8 Quantization\n    teacher_model.cpu().eval()\n    student_model.cpu().eval()\n\n    student_dir = MODELS_DIR / "student_fp32"\n    student_model.save_pretrained(student_dir)\n\n    quantized_student = torch.ao.quantization.quantize_dynamic(\n        student_model, {nn.Linear}, dtype=torch.qint8\n    )\n    quantized_path = MODELS_DIR / "quantized_student_int8.pt"\n    torch.save(quantized_student, quantized_path)\n\n    # =========================================================================\n    # 5. Evaluate All 4 Models on the Same Held-Out Test Set (CPU)\n    # =========================================================================\n    print("Evaluating all 4 models on the held-out test set...")\n\n    def pred_m1(text):\n        return int(clf.predict(vectorizer.transform([text]))[0])\n\n    def pred_m2(text):\n        vec = torch.tensor(sentence_vector(text.split(), w2v), dtype=torch.float32).unsqueeze(0)\n        with torch.no_grad():\n            return int(torch.argmax(ffnn(vec), dim=-1).item())\n\n    def pred_m3(text):\n        enc = tokenizer(text, return_tensors="pt", truncation=True, max_length=48)\n        with torch.no_grad():\n            return int(torch.argmax(teacher_model(**enc).logits, dim=-1).item())\n\n    def pred_m4(text):\n        enc = tokenizer(text, return_tensors="pt", truncation=True, max_length=48)\n        with torch.no_grad():\n            # DistilBERT has no token_type_ids -> pass only the inputs it accepts\n            logits = quantized_student(input_ids=enc["input_ids"], attention_mask=enc["attention_mask"]).logits\n            return int(torch.argmax(logits, dim=-1).item())\n\n    def pred_m4_fp32(text):\n        enc = tokenizer(text, return_tensors="pt", truncation=True, max_length=48)\n        with torch.no_grad():\n            logits = student_model(input_ids=enc["input_ids"], attention_mask=enc["attention_mask"]).logits\n            return int(torch.argmax(logits, dim=-1).item())\n\n    m1_res = measure_metrics_and_latency(pred_m1, X_test_text, y_test, tfidf_path)\n    m2_res = measure_metrics_and_latency(pred_m2, X_test_text, y_test, ffnn_path)\n    m3_res = measure_metrics_and_latency(pred_m3, X_test_text, y_test, teacher_dir)\n    m4_res = measure_metrics_and_latency(pred_m4, X_test_text, y_test, quantized_path)\n    # Extra row (not a required model): student before quantization, to isolate the effect of INT8\n    m4f_res = measure_metrics_and_latency(pred_m4_fp32, X_test_text, y_test, student_dir)\n\n    results = {\n        "hardware": hardware_description(),\n        "n_train": len(train_raw), "n_val": len(val_raw), "n_test": len(test_raw),\n        "latency_runs": 100,\n        "models": [\n            {"model": "1. TF-IDF + Logistic Regression", "accuracy": m1_res[0], "oos_recall": m1_res[1], "f1": m1_res[2], "size_mb": m1_res[3], "latency_ms": m1_res[4]},\n            {"model": "2. Word2Vec + SGD Feedforward NN", "accuracy": m2_res[0], "oos_recall": m2_res[1], "f1": m2_res[2], "size_mb": m2_res[3], "latency_ms": m2_res[4]},\n            {"model": "3. Fine-Tuned BERT (Teacher FP32)", "accuracy": m3_res[0], "oos_recall": m3_res[1], "f1": m3_res[2], "size_mb": m3_res[3], "latency_ms": m3_res[4]},\n            {"model": "4a. Distilled DistilBERT (FP32, before quantization) [extra]", "accuracy": m4f_res[0], "oos_recall": m4f_res[1], "f1": m4f_res[2], "size_mb": m4f_res[3], "latency_ms": m4f_res[4]},\n            {"model": "4. Distilled + Quantized BERT (INT8)", "accuracy": m4_res[0], "oos_recall": m4_res[1], "f1": m4_res[2], "size_mb": m4_res[3], "latency_ms": m4_res[4]}\n        ]\n    }\n\n    with open("results.json", "w") as f:\n        json.dump(results, f, indent=2)\n\n    print("\\n--- Final Results Table (R4) ---")\n    for row in results["models"]:\n        print(row)\n\n\nif __name__ == "__main__":\n    main()\n')
print('Project scripts created.')


In [ ]:
# Train all 4 models and print the R4 Results Table
!python train_models.py

In [ ]:
import json
print(json.dumps(json.load(open('results.json')), indent=2))


In [ ]:
!zip -r models_trained.zip models results.json
from google.colab import files
files.download('models_trained.zip')
